# ESM-2 as a Surrogate Model for Protein Active Learning

This tutorial introduces the ESM-2 protein language model as a frozen encoder and LoRA-finetuned surrogate for active learning. We use the GFP fluorescence dataset as the primary vehicle, with a ProteinGym DMS assay as a real-world extension.

**Prerequisites:** Complete these tutorials first:
- `tutorials/experiments/offline_design_tutorial.ipynb`
- `tutorials/models/gp_tutorial.ipynb`

**What you will learn:**
1. How ESM-2 embeddings encode protein function without task-specific training
2. How to train a regression head on frozen vs LoRA-adapted ESM-2
3. How MC Dropout provides calibrated uncertainty estimates
4. How to run a full active learning cycle with ESM-2 + UCB acquisition
5. How to transfer ESM-2 to a DMS fitness landscape (ProteinGym)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import scipy.stats
from sklearn.decomposition import PCA
from IPython.display import Image, display
from pathlib import Path

from alf_core import (
    BaseDatasetConfig,
    DatasetSearch,
    DesignTask,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_tools.datasets import GFP
from alf_tools.models.esm2 import (
    ESM2DropoutModel,
    ESM2ModelConfig,
    ESM2TrainConfig,
    ESM2RegressionHead,
)
from alf_tools.models.gp import GPModel, GPModelConfig, FeaturizerConfig
from alf_tools.models.utils import extract_sequences_from_inputs
from alf_tools.optimizer.acquisition_functions import UCB

# Device detection: GPU if available, CPU fallback
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️  Running on CPU — long-running cells will be slower. "
          "See timing notes in each section.")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED);
print("Setup complete.")

---
## Section 1: ESM-2 as a Sequence Encoder

**Goals:**
- Load the GFP dataset and inspect sequence/label distributions
- Compute ESM-2 mean-pooled embeddings for a subset of sequences
- Visualise the embedding space with PCA, coloured by fitness
- Understand why ESM-2 embeddings are valuable compared to raw sequence features

ESM-2 was pretrained on ~250M protein sequences from UniRef50. Even before any task-specific training, its 640-dimensional embeddings capture evolutionary relationships and functional properties of proteins.

In [ ]:
# GFP dataset split (used in Sections 1 and 2): 480 train, 120 val, 200 test, 200 pool
gfp_supervised = GFP(BaseDatasetConfig(
    name="gfp",
    modality="sequence",
    seed=SEED,
    train_ratio=0.6,       # 600 sequences for train+val
    validation_frac=0.2,   # 20% of train+val → 120 val, 480 train
    test_ratio=0.2,        # 200 test sequences
    split_type="random",
))
gfp_supervised.setup()

train_data = gfp_supervised.train_dataset
val_data   = gfp_supervised.validation_dataset
test_data  = gfp_supervised.test_dataset

print(gfp_supervised)

In [ ]:
# Verify expected split sizes
assert len(train_data) == 480, f"Expected 480 train, got {len(train_data)}"
assert len(val_data)   == 120, f"Expected 120 val, got {len(val_data)}"
assert len(test_data)  == 200, f"Expected 200 test, got {len(test_data)}"

# Labels are continuous brightness values
labels_all = np.array(train_data.labels)
assert labels_all.ndim == 1
print(f"Train label range: [{labels_all.min():.3f}, {labels_all.max():.3f}]")
print("Sample sequences (first 3):")
for cand in list(train_data.candidates)[:3]:
    print(f"  {cand.data[:30]}...")
print("✓ GFP dataset loaded and splits verified")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label histogram
for split, data, colour in [
    ("train", train_data, "steelblue"),
    ("val",   val_data,   "orange"),
    ("test",  test_data,  "green"),
]:
    axes[0].hist(np.array(data.labels), bins=30, alpha=0.6, label=split, color=colour)
axes[0].set_xlabel("Median Brightness"); axes[0].set_ylabel("Count")
axes[0].set_title("GFP Brightness Distribution by Split")
axes[0].legend()

# Sequence length distribution
lengths = [len(c.data) for c in list(train_data.candidates)]
axes[1].hist(lengths, bins=20, color="steelblue", alpha=0.8)
axes[1].set_xlabel("Sequence Length"); axes[1].set_ylabel("Count")
axes[1].set_title("GFP Sequence Length Distribution (Train)")

plt.tight_layout(); plt.show()